# 01 - Environment setup

**Project:** Edge-Aware Multi-Scale Autoencoder for Unsupervised Industrial Anomaly Detection
**Phase:** 1 - Foundations

**Purpose:** verify the Colab runtime (Python, PyTorch, GPU), install missing dependencies only, make the `src` package importable, set reproducibility seeds, and confirm every check passes before moving on to the dataset notebook.

## How to run (VS Code + Google Colab extension)

1. Open this notebook in VS Code, then connect the kernel: **Select Kernel -> Colab -> Auto Connect** and sign in with your Google account when prompted.
2. When provisioning a server, choose a **GPU** machine type (required later for training).
3. **Before the first run:** in the VS Code Explorer, right-click the repository folder (`edge-aware-anomaly-detection`) -> **Upload to Colab**. The kernel runs on Google's servers and cannot see your local disk; uploading is what makes `import src` possible.
   - If your local `data/` or `outputs/` folders already contain large files, multi-select just `src/`, `requirements.txt` and `.gitignore` instead of the whole folder.
4. Run all cells top to bottom (**Run All**). Stop and fix the problem if the final checklist reports any FAIL.

> After you edit files inside `src/`, re-upload the repository folder. The runtime keeps its own copy, and the import cell prints a checksum so you can tell which copy is loaded.


In [2]:
from pathlib import Path

REPO_DIR_NAME = "edge-aware-anomaly-detection"

# Candidate repository roots on this runtime, in priority order:
#   1. cwd                  - covers local kernels or a mounted server workspace
#   2. /content/<repo>      - whole repo uploaded via right-click -> Upload to Colab
#   3. Google Drive copy    - only if Drive was mounted and holds a repo copy
#   4. /content             - fallback when only the src/ folder was uploaded
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content") / REPO_DIR_NAME,
    Path("/content/drive/MyDrive") / REPO_DIR_NAME,
    Path("/content"),
]


def _is_repo_root(path: Path) -> bool:
    return (path / "src" / "config.py").is_file() and (path / "src" / "__init__.py").is_file()


print("Probe results:")
hits = []
for root in CANDIDATE_ROOTS:
    hit = _is_repo_root(root)
    print(f"  {'HIT ' if hit else 'miss'}  {root}")
    if hit:
        hits.append(root)

if not hits:
    raise FileNotFoundError(
        "src/config.py not found on this runtime.\n"
        "Fix: in the VS Code Explorer, right-click the repository folder "
        f"('{REPO_DIR_NAME}') -> 'Upload to Colab', then re-run this cell."
    )

REPO_ROOT = hits[0]
if len(hits) > 1:
    print(f"\nMultiple hits; using the first: {REPO_ROOT}")

# The checks below also need requirements.txt and .gitignore, not just src/.
missing = [
    name for name in ("requirements.txt", ".gitignore") if not (REPO_ROOT / name).is_file()
]
if missing:
    raise FileNotFoundError(
        f"Found src/ under {REPO_ROOT}, but {missing} is missing there.\n"
        "Only part of the repository was uploaded. Upload the WHOLE repository "
        f"folder ('{REPO_DIR_NAME}'): right-click it -> 'Upload to Colab'."
    )

print(f"\nRepo root on runtime: {REPO_ROOT}")


Probe results:
  miss  /content
  HIT   /content/edge-aware-anomaly-detection
  miss  /content/drive/MyDrive/edge-aware-anomaly-detection
  miss  /content

Repo root on runtime: /content/edge-aware-anomaly-detection


In [3]:
import datetime
import sys
from pathlib import Path

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src
from src import config

src_file = Path(src.__file__).resolve()
repo_resolved = REPO_ROOT.resolve()
assert src_file.is_relative_to(repo_resolved), (
    f"Imported 'src' from {src_file}, which is not under {repo_resolved}. "
    "A different 'src' package is shadowing the project module - restart the "
    "kernel and re-run from the top."
)

cfg_file = Path(config.__file__).resolve()
mtime = datetime.datetime.fromtimestamp(cfg_file.stat().st_mtime)

print(f"src package       : {src_file}")
print(f"src.__version__   : {src.__version__}")
print(f"config module     : {cfg_file}")
print(f"config.py sha256  : {config.file_sha256(cfg_file)} (first 12 hex chars)")
print(f"config.py mtime   : {mtime:%Y-%m-%d %H:%M:%S} (runtime clock)")
print(
    "\nStaleness guard: if you edit files under src/ locally, re-upload the "
    "repository folder to this runtime and compare the checksum above."
)

SRC_OK = True


src package       : /content/edge-aware-anomaly-detection/src/__init__.py
src.__version__   : 0.1.0
config module     : /content/edge-aware-anomaly-detection/src/config.py
config.py sha256  : c0d6f69cecfa (first 12 hex chars)
config.py mtime   : 2026-10-14 14:43:28 (runtime clock)

Staleness guard: if you edit files under src/ locally, re-upload the repository folder to this runtime and compare the checksum above.


## Runtime checks: Python, PyTorch, GPU

Colab images change over time, so nothing here is assumed. We *measure* the environment: Python version, torch/torchvision versions, CUDA availability, and the GPU model. The GPU check is a hard failure when the kernel is a Colab runtime (a missing GPU there almost always means the server was created without one) but only a warning on a local kernel.


In [4]:
import sys

MIN_PY = (3, 9)

print(f"Python     : {sys.version.split()[0]}")
print(f"Executable : {sys.executable}")

py_ok = sys.version_info >= MIN_PY
print(f"Required   : >= {MIN_PY[0]}.{MIN_PY[1]}  ->  {'PASS' if py_ok else 'FAIL'}")
print("Note: any Python >= 3.9 is fine; current Colab images ship 3.10-3.12.")

assert py_ok, f"Python {sys.version.split()[0]} is older than the minimum supported version."


Python     : 3.13.15
Executable : /usr/bin/python3
Required   : >= 3.9  ->  PASS
Note: any Python >= 3.9 is fine; current Colab images ship 3.10-3.12.


In [5]:
import importlib.util
import subprocess

in_colab = importlib.util.find_spec("google.colab") is not None
print(f"Colab runtime detected: {in_colab}")

import torch
import torchvision

print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"torch CUDA  : {torch.version.cuda}")
print(f"cuDNN       : {torch.backends.cudnn.version()}")

gpu_ok = torch.cuda.is_available()
print(f"\ntorch.cuda.is_available(): {gpu_ok}")
if gpu_ok:
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        total_gib = props.total_memory / 1024**3
        print(f"  [{index}] {props.name} | capability {props.major}.{props.minor} | {total_gib:.1f} GiB")
else:
    print("No GPU visible to this kernel.")

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("\nnvidia-smi (first lines):")
        print("\n".join(result.stdout.splitlines()[:12]))
    else:
        print("\nnvidia-smi returned non-zero; skipping extra GPU details.")
except (FileNotFoundError, subprocess.TimeoutExpired) as error:
    print(f"\nnvidia-smi unavailable ({error}); skipping extra GPU details.")

if in_colab:
    assert gpu_ok, (
        "No GPU on this Colab runtime. Run 'Colab: Remove Server' from the "
        "command palette, provision a new server and choose a GPU machine type."
    )
elif not gpu_ok:
    print("\nWARNING: no GPU on this kernel. Fine for setup checks; training needs Colab.")


Colab runtime detected: True
torch       : 2.11.0+cu130
torchvision : 0.26.0+cu130
torch CUDA  : 13.0
cuDNN       : 92700

torch.cuda.is_available(): True
  [0] Tesla T4 | capability 7.5 | 14.6 GiB

nvidia-smi (first lines):
Wed Oct 14 14:44:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A  

## Dependencies: install only what is missing

`requirements.txt` is the source of truth. For each entry we check whether the package is already importable and print its installed version. Only genuinely missing packages are installed.

**torch and torchvision are special:** Colab ships a CUDA-matched build. Reinstalling it with plain `pip` typically destroys the runtime's GPU support, so they are reported but never (re)installed by this cell.


In [6]:
import importlib
import importlib.metadata as metadata
import importlib.util
import re
import subprocess
import sys
from pathlib import Path

REQ_FILE = REPO_ROOT / "requirements.txt"

# requirement name -> importable module name (they differ for a few packages)
IMPORT_NAME = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "pillow": "PIL",
    "pyyaml": "yaml",
    "tqdm": "tqdm",
    "pandas": "pandas",
}

# Colab ships a CUDA-matched torch build; reinstalling it would break the GPU.
NEVER_REINSTALL = {"torch", "torchvision"}


def parse_requirements(path: Path):
    specs = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if line:
            specs.append(line)
    return specs


def package_name(spec: str) -> str:
    return re.split(r"[<>=!~\[;]", spec, maxsplit=1)[0].strip()


def installed_version(name: str):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def module_of(name: str) -> str:
    return IMPORT_NAME.get(name.lower(), name.lower().replace("-", "_"))


specs = parse_requirements(REQ_FILE)
missing = []

print(f"{'package':<14}{'requirement':<15}{'installed':<12}status")
print("-" * 58)
for spec in specs:
    name = package_name(spec)
    present = importlib.util.find_spec(module_of(name)) is not None
    version = installed_version(name) or "-"
    if present:
        status = "ok"
    elif name.lower() in NEVER_REINSTALL:
        status = "MISSING (install manually - never auto-reinstall)"
        missing.append(name)  # reported, but see NEVER_REINSTALL below
    else:
        status = "MISSING"
        missing.append(name)
    print(f"{name:<14}{spec:<15}{version:<12}{status}")

auto_install = [name for name in missing if name.lower() not in NEVER_REINSTALL]
manual = [name for name in missing if name.lower() in NEVER_REINSTALL]

print()
if auto_install:
    print(f"Installing only the missing packages: {' '.join(auto_install)}")
    try:
        shell = get_ipython()  # noqa: F821 - defined in IPython/Colab kernels
    except NameError:
        shell = None
    if shell is not None:
        shell.run_line_magic("pip", "install -q " + " ".join(auto_install))
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *auto_install], check=True)
    importlib.invalidate_caches()
elif manual:
    print(f"torch/torchvision missing: {manual}. On Colab this is unexpected - reconnect the runtime.")
else:
    print("All required packages are already present; nothing to install.")

unsatisfied = [name for name in (package_name(spec) for spec in specs)
               if importlib.util.find_spec(module_of(name)) is None]
REQS_OK = not unsatisfied
if unsatisfied:
    print(f"\nStill unsatisfied: {unsatisfied}")


package       requirement    installed   status
----------------------------------------------------------
torch         torch>=2.0     2.11.0+cu130ok
torchvision   torchvision>=0.150.26.0+cu130ok
numpy         numpy>=1.24    2.1.3       ok
matplotlib    matplotlib>=3.73.10.0      ok
pillow        pillow>=9.0    11.3.0      ok
pyyaml        pyyaml>=6.0    6.0.3       ok
tqdm          tqdm>=4.65     4.67.3      ok
pandas        pandas>=2.0    2.2.3       ok

All required packages are already present; nothing to install.


## Reproducibility

Every experiment in this project starts from the same seed. `set_seed()` (in `src/config.py`) seeds Python's `random`, the `PYTHONHASHSEED` environment variable, NumPy if present, and PyTorch/CUDA if present, and forces cuDNN toward deterministic kernels. Determinism is a research requirement here: differences between runs must come from our methods, not from unseeded randomness.


In [7]:
from src.config import SEED, set_seed

applied_seed = set_seed(SEED)
SEED_OK = applied_seed == SEED

print(f"Global seed applied: {applied_seed}")
print("Seeded: python 'random', PYTHONHASHSEED, numpy (if installed), torch (+ CUDA).")
print("cuDNN: deterministic=True, benchmark=False (reproducibility over speed).")
print(
    "Note: bitwise-identical GPU results are not guaranteed for every operation; "
    "we accept small residual nondeterminism in exchange for reproducible research runs."
)


Global seed applied: 42
Seeded: python 'random', PYTHONHASHSEED, numpy (if installed), torch (+ CUDA).
cuDNN: deterministic=True, benchmark=False (reproducibility over speed).
Note: bitwise-identical GPU results are not guaranteed for every operation; we accept small residual nondeterminism in exchange for reproducible research runs.


## Workspace directories and git hygiene

`outputs/` (checkpoints, plots, logs) and `data/` (the MVTec AD dataset) can grow to gigabytes. They are created here so every later notebook agrees on their location, and `.gitignore` must cover them so they can never be committed - the dataset has its own license and checkpoints do not belong in a portfolio repository.


In [8]:
from src.config import get_paths

paths = get_paths()
print(f"repo root: {paths.repo_root}")

for directory in (paths.outputs, paths.data):
    directory.mkdir(parents=True, exist_ok=True)
    print(f"ready: {directory}")
DIRS_OK = paths.outputs.is_dir() and paths.data.is_dir()

gitignore_path = paths.repo_root / ".gitignore"
gitignore_text = gitignore_path.read_text(encoding="utf-8")
required_patterns = ["data/", "outputs/", "__pycache__/", ".ipynb_checkpoints/"]
GI_OK = True
print(f"\n.gitignore protection ({gitignore_path}):")
for pattern in required_patterns:
    covered = any(
        pattern in line or pattern.rstrip("/") in line
        for line in gitignore_text.splitlines()
    )
    print(f"  {'ok  ' if covered else 'MISS'}  {pattern}")
    GI_OK = GI_OK and covered
assert GI_OK, ".gitignore is missing required patterns - large files could end up in git."

print("\nTop-level entries of the repo copy used by this runtime:")
for item in sorted(paths.repo_root.iterdir()):
    print(f"  {item.name}/" if item.is_dir() else f"  {item.name}")

print(
    "\nNote: the dataset location can be redirected later with the MVTEC_ROOT "
    "environment variable, e.g. to a mounted Google Drive folder."
)


repo root: /content/edge-aware-anomaly-detection
ready: /content/edge-aware-anomaly-detection/outputs
ready: /content/edge-aware-anomaly-detection/data

.gitignore protection (/content/edge-aware-anomaly-detection/.gitignore):
  ok    data/
  ok    outputs/
  ok    __pycache__/
  ok    .ipynb_checkpoints/

Top-level entries of the repo copy used by this runtime:
  .env
  .env.example
  .git/
  .gitignore
  README.md
  data/
  notebooks/
  outputs/
  requirements.txt
  src/

Note: the dataset location can be redirected later with the MVTEC_ROOT environment variable, e.g. to a mounted Google Drive folder.


## Final checklist

Every line must read PASS. A FAIL here means the runtime is not ready - fix it before touching the dataset or any future training code.


In [9]:
import importlib.util

checks = {
    "Python >= 3.9": py_ok,
    "src importable from repo copy": SRC_OK,
    "requirements satisfied": REQS_OK,
    "torch importable": importlib.util.find_spec("torch") is not None,
    "GPU visible (hard requirement on Colab)": (gpu_ok if in_colab else True),
    "seed applied": SEED_OK,
    "outputs/ and data/ created": DIRS_OK,
    ".gitignore protects them": GI_OK,
}

width = max(len(name) for name in checks)
all_ok = True
for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name.ljust(width)}")
    all_ok = all_ok and bool(ok)

assert all_ok, "Some checks failed - fix them before continuing."
print("\nAll checks passed. Next: 02_dataset_inspection.ipynb (not created yet).")


[PASS] Python >= 3.9                          
[PASS] src importable from repo copy          
[PASS] requirements satisfied                 
[PASS] torch importable                       
[PASS] GPU visible (hard requirement on Colab)
[PASS] seed applied                           
[PASS] outputs/ and data/ created             
[PASS] .gitignore protects them               

All checks passed. Next: 02_dataset_inspection.ipynb (not created yet).


## Keeping your work safe when the runtime dies

A Colab runtime is a temporary virtual machine. When it disconnects (idle timeout, error, manual reset), **everything under `/content` is deleted**: uploaded repo copies, downloaded datasets, checkpoints, pip-installed packages.

Your safety layers:

1. **Code and notebooks** - they live on your disk in VS Code. The kernel only executes them; it never owns them. A dead runtime cannot lose them. When ready, push to your GitHub repository for versioning and portfolio use.
2. **Cell outputs** - saved into your local `.ipynb` when cells run, so tables and plots survive reconnects (re-run on a fresh runtime to refresh them).
3. **Large or costly artifacts** (datasets, checkpoints, logs) - keep them on **Google Drive**, never only in `/content`:
   - Command palette -> `Colab: Mount Google Drive to Server...` (appends a mount cell), or the *Colab* button in the notebook toolbar;
   - save under `/content/drive/MyDrive/edge-aware-anomaly-detection/...`;
   - point the dataset at Drive later via the `MVTEC_ROOT` environment variable (see `src/config.py`).

**Rule of thumb:** if losing a file would hurt, it must exist on your disk, on GitHub, or on Drive - never only on the runtime.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `FileNotFoundError: src/config.py not found` | Kernel cannot see your local disk | Right-click the repository folder in the VS Code Explorer -> **Upload to Colab**; re-run the locate cell |
| "...requirements.txt is missing" error | Only the `src/` folder was uploaded | Upload the whole repository folder (or multi-select `src/`, `requirements.txt`, `.gitignore`) |
| Assertion "not under repo root" / odd `ModuleNotFoundError` | Another package named `src` shadows the project | Restart the kernel and run the first two cells first; check the printed `src.__file__` |
| Edited `src/`, but changes have no effect | Runtime still holds the old upload | Re-upload; compare the printed sha256 checksum with your local file |
| `torch.cuda.is_available()` is False on Colab | Server was created without a GPU | `Colab: Remove Server`, provision a new server, pick a GPU machine type |
| Torch broken right after a pip install | A reinstall replaced Colab's CUDA build | Never pip-install torch/torchvision on Colab; disconnect and delete the runtime, reconnect |
| pip installs run on every session | Runtime was reset, wiping site-packages | Normal after reconnect; the audit cell only installs what is actually missing |
| Drive mount asks for permission | First mount needs OAuth consent | Run the appended mount cell and allow access in the browser popup |
